# Utility: Comment-Driven Quality Control Engine

This engine scans the **Catalog Metadata (Column Comments)** to find quality rules. 
If a column has a comment like `DQ: NOT NULL`, this engine will automatically enforce it.

In [0]:
from pyspark.sql.functions import col, lit, expr, concat_ws

def apply_metadata_dq(df, table_name):
    """
    Scans the Data Catalog for comments on 'table_name'.
    Parses comments starting with 'DQ:' to apply rules.
    """
    # 1. Initialize DQ columns
    validated_df = df.withColumn("is_invalid", lit(False)) \
                     .withColumn("dq_errors", lit(""))

    try:
        # 2. Get Column metadata from the Catalog
        columns = spark.catalog.listColumns(table_name)
    except:
        print(f"⚠ Table {table_name} not found in catalog. Skipping metadata DQ.")
        return validated_df

    found_rules = False
    for c in columns:
        # Check if the comment contains our DQ trigger
        if c.description and "DQ:" in c.description:
            found_rules = True
            # Extract rules from the comment, e.g., "DQ: NOT NULL, > 0"
            rule_part = c.description.split("DQ:")[1]
            rules = [r.strip() for r in rule_part.split(",")]
            
            for rule in rules:
                condition = None
                if rule.upper() == "NOT NULL":
                    condition = f"{c.name} IS NOT NULL"
                elif "> 0" in rule:
                    condition = f"{c.name} > 0"
                elif ">= 0" in rule:
                    condition = f"{c.name} >= 0"
                
                if condition:
                    # Apply the dynamic rule
                    validated_df = validated_df.withColumn("is_invalid", 
                        expr(f"CASE WHEN NOT ({condition}) THEN true ELSE is_invalid END")
                    ).withColumn("dq_errors", 
                        expr(f"CASE WHEN NOT ({condition}) THEN concat_ws('; ', dq_errors, '{c.name}_{rule}') ELSE dq_errors END")
                    )
    
    if not found_rules:
        print(f"ℹ No 'DQ:' tags found in comments for table {table_name}.")
        
    return validated_df